# Day 083 — Solution: A Planning Agent

In [ ]:
_SRC = '"""planner_agent.py — Day 083: Planning & Decomposition.\n\nDays 79-82 gave an agent memory of what was just said and who it is talking to.\nBut every day so far the agent has worked on one task at a time. This day teaches\ndecomposition: given a complex goal, the agent breaks it into an ordered list of\nsubtasks, resolves their dependencies, and executes them step by step.\n\nPieces:\n  safe_parse_json / call_llm  - reused (Day 79)\n  safe_parse_list              - list-aware variant of safe_parse_json\n  Task                         - a dataclass: id, title, description, depends_on\n  build_plan_prompt            - ask the LLM to decompose a goal into tasks\n  parse_plan                   - extract the task list from LLM output (never raises)\n  topo_sort                    - Kahn\'s topological sort: resolve dependency order\n  build_execution_context      - inject prior task results into the current prompt\n  execute_task                 - run one task; uses executor_fn or the LLM\n  run_plan                     - plan -> sort -> execute; max_tasks safety guard\n  PlannerAgent                 - an agent that plans, sorts, and executes\n\nSetup:\n    pip install ollama\n    ollama pull llama3.2\n"""\nimport json\nfrom dataclasses import dataclass, field\n\n# ── helpers reused from Day 79 ───────────────────────────────────────────────\ndef safe_parse_json(text):\n    """Slice first \'{\' to last \'}\' and parse. Returns dict|None (Day 79)."""\n    start, end = text.find("{"), text.rfind("}")\n    if start == -1 or end == -1 or end < start:\n        return None\n    try:\n        data = json.loads(text[start:end + 1])\n    except (json.JSONDecodeError, ValueError):\n        return None\n    return data if isinstance(data, dict) else None\n\n\ndef safe_parse_list(text):\n    """Slice first \'[\' to last \']\' and parse. Returns list|None. Never raises."""\n    start, end = text.find("["), text.rfind("]")\n    if start == -1 or end == -1 or end < start:\n        return None\n    try:\n        data = json.loads(text[start:end + 1])\n    except (json.JSONDecodeError, ValueError):\n        return None\n    return data if isinstance(data, list) else None\n\n\ndef call_llm(messages, llm_fn=None):\n    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""\n    if llm_fn is not None:\n        return llm_fn(messages)\n    import ollama\n    resp = ollama.chat(model="llama3.2", messages=messages)\n    return resp["message"]["content"]\n\n# ── the Task dataclass ────────────────────────────────────────────────────────\n@dataclass\nclass Task:\n    """One step in a plan.\n\n    Attributes:\n        id:          short snake_case identifier (e.g. \'t1\', \'write_outline\').\n        title:       brief human-readable label (5 words max).\n        description: one sentence describing what to do.\n        depends_on:  ids of tasks that must complete before this one.\n        status:      \'pending\' | \'done\' | \'failed\'.\n        result:      the output of executing this task.\n    """\n    id: str\n    title: str\n    description: str\n    depends_on: list = field(default_factory=list)\n    status: str = "pending"\n    result: str = ""\n\n# ── planning: ask the LLM to decompose a goal ─────────────────────────────────\ndef build_plan_prompt(goal, context=None):\n    """Build a prompt that asks the LLM to break a goal into a JSON task list."""\n    system = "\\n".join([\n        "You are a planning assistant. Break the goal into an ordered list of tasks.",\n        "",\n        "Return ONLY a JSON array. Each item must have these exact keys:",\n        \'  "id": short snake_case identifier (t1, t2, ...)\',\n        \'  "title": brief label (5 words max)\',\n        \'  "description": one sentence - what to do\',\n        \'  "depends_on": list of task ids that must finish before this one ([] if none)\',\n        "",\n        "Return ONLY the JSON array. No prose, no markdown fences.",\n    ])\n    user_parts = ["Goal: " + str(goal)]\n    if context:\n        user_parts.append("Context: " + str(context))\n    return [{"role": "system", "content": system},\n            {"role": "user", "content": "\\n".join(user_parts)}]\n\n\ndef parse_plan(text):\n    """Extract a task list from LLM output. Returns list[Task]; never raises.\n\n    Tolerates markdown fences, prose before/after, missing fields, and invalid\n    JSON. Invalid or missing fields are filled with safe defaults so any\n    parseable item becomes a valid Task.\n    """\n    items = safe_parse_list(text) or []\n    tasks = []\n    for i, item in enumerate(items):\n        if not isinstance(item, dict):\n            continue\n        tasks.append(Task(\n            id=str(item.get("id", "t" + str(i + 1))),\n            title=str(item.get("title", "Task " + str(i + 1))),\n            description=str(item.get("description", "")),\n            depends_on=[str(d) for d in item.get("depends_on", [])\n                        if isinstance(d, str)],\n        ))\n    return tasks\n\n# ── dependency ordering: Kahn\'s topological sort ──────────────────────────────\ndef topo_sort(tasks):\n    """Sort tasks so every dependency comes before the task that needs it.\n\n    Uses Kahn\'s algorithm (BFS on a DAG). If a cycle exists the cyclic tasks\n    are appended at the end in their original order rather than raising, so\n    execution can still proceed on the non-cyclic portion.\n    """\n    by_id = {t.id: t for t in tasks}\n    # count incoming edges (how many unresolved deps each task has)\n    in_deg = {t.id: 0 for t in tasks}\n    for t in tasks:\n        for dep in t.depends_on:\n            if dep in in_deg:\n                in_deg[t.id] += 1\n    # start with tasks that have no deps\n    queue = [t.id for t in tasks if in_deg[t.id] == 0]\n    order = []\n    while queue:\n        tid = queue.pop(0)\n        order.append(by_id[tid])\n        # for every task that listed tid as a dep, reduce its in-degree\n        for t in tasks:\n            if tid in t.depends_on:\n                in_deg[t.id] -= 1\n                if in_deg[t.id] == 0:\n                    queue.append(t.id)\n    # cycle guard: any task not yet emitted has a circular dependency\n    done_ids = {t.id for t in order}\n    for t in tasks:\n        if t.id not in done_ids:\n            order.append(t)\n    return order\n\n# ── executing tasks ───────────────────────────────────────────────────────────\ndef build_execution_context(task, results):\n    """Render prior results that this task depends on, for injection into the prompt."""\n    lines = ["You are executing one step of a multi-task plan."]\n    prior = [(dep, results[dep]) for dep in task.depends_on if dep in results]\n    if prior:\n        lines.append("Results from earlier steps:")\n        for dep_id, res in prior:\n            lines.append("  " + dep_id + ": " + str(res))\n    lines.append("Task: " + task.title)\n    lines.append("Description: " + task.description)\n    lines.append("Complete this task concisely.")\n    return "\\n".join(lines)\n\n\ndef execute_task(task, results, executor_fn=None, llm_fn=None):\n    """Run one task. Returns the result string; never raises.\n\n    If executor_fn is provided, call executor_fn(task) -> str.\n    Otherwise use the LLM, passing prior results as context.\n    Exceptions are caught and returned as \'Error: ...\' strings.\n    """\n    try:\n        if executor_fn is not None:\n            return str(executor_fn(task))\n        context = build_execution_context(task, results)\n        messages = [{"role": "system", "content": context},\n                    {"role": "user", "content": "Execute this task now."}]\n        return call_llm(messages, llm_fn=llm_fn)\n    except Exception as exc:\n        return "Error: " + str(exc)\n\n# ── end-to-end plan runner ────────────────────────────────────────────────────\ndef run_plan(goal, executor_fn=None, llm_fn=None, max_tasks=20):\n    """Plan a goal, sort by dependencies, and execute step by step.\n\n    Returns {"tasks": list[Task], "results": {id: result}, "answer": str}.\n    The answer is the result of the last task in execution order.\n    max_tasks caps the plan so a model that returns 1000 tasks cannot hang the gate.\n    """\n    plan_text = call_llm(build_plan_prompt(goal), llm_fn=llm_fn)\n    tasks = parse_plan(plan_text)\n    if not tasks:\n        return {"tasks": [], "results": {}, "answer": "No plan generated."}\n    ordered = topo_sort(tasks[:max_tasks])\n    results = {}\n    for task in ordered:\n        result = execute_task(task, results,\n                              executor_fn=executor_fn, llm_fn=llm_fn)\n        task.result = result\n        task.status = "done"\n        results[task.id] = result\n    answer = ordered[-1].result if ordered else "No tasks executed."\n    return {"tasks": ordered, "results": results, "answer": answer}\n\n# ── the planning assistant ────────────────────────────────────────────────────\nclass PlannerAgent:\n    """An agent that breaks a goal into subtasks and executes them in order.\n\n    plan() decomposes a goal without executing - useful for reviewing the plan.\n    execute() runs a task list produced by plan() (or built manually).\n    run() does both in one call and records the outcome in history.\n\n    Example::\n\n        agent = PlannerAgent(executor_fn=my_executor, llm_fn=my_llm_fn)\n        result = agent.run("Write a short report on prompt engineering")\n        print(result["answer"])\n    """\n\n    def __init__(self, executor_fn=None, llm_fn=None, max_tasks=20):\n        self._executor_fn = executor_fn\n        self._llm_fn = llm_fn\n        self.max_tasks = max_tasks\n        self._history = []\n\n    def plan(self, goal):\n        """Decompose goal into a topologically sorted task list; do not execute."""\n        plan_text = call_llm(build_plan_prompt(goal), llm_fn=self._llm_fn)\n        return topo_sort(parse_plan(plan_text)[:self.max_tasks])\n\n    def execute(self, tasks):\n        """Execute a task list in dependency order. Returns {id: result} dict."""\n        ordered = topo_sort(tasks[:self.max_tasks])\n        results = {}\n        for task in ordered:\n            result = execute_task(task, results,\n                                  executor_fn=self._executor_fn,\n                                  llm_fn=self._llm_fn)\n            task.result = result\n            task.status = "done"\n            results[task.id] = result\n        return results\n\n    def run(self, goal):\n        """Plan + execute. Records the run in history and returns the result dict."""\n        tasks = self.plan(goal)\n        results = self.execute(tasks)\n        answer = tasks[-1].result if tasks else "No tasks."\n        record = {"goal": goal, "tasks": tasks, "results": results, "answer": answer}\n        self._history.append(record)\n        return record\n\n    def history(self):\n        """Return a copy of the run history."""\n        return list(self._history)\n\n    def clear_history(self):\n        """Clear run history in place."""\n        self._history.clear()\n'
from pathlib import Path
Path('planner_agent.py').write_text(_SRC, encoding='utf-8')
print('planner_agent.py written.')

In [ ]:

import json
from planner_agent import (
    Task, safe_parse_list, build_plan_prompt, parse_plan,
    topo_sort, build_execution_context, execute_task,
    run_plan, PlannerAgent,
)

_PLAN_JSON = json.dumps([
    {'id': 't1', 'title': 'Gather facts',
     'description': 'Collect the relevant information.', 'depends_on': []},
    {'id': 't2', 'title': 'Draft outline',
     'description': 'Organize the facts into an outline.', 'depends_on': ['t1']},
    {'id': 't3', 'title': 'Write summary',
     'description': 'Write the final summary.', 'depends_on': ['t2']},
])

def _mock_planner(plan_json=None, task_result='Task done.'):
    plan = plan_json if plan_json is not None else _PLAN_JSON
    def _fn(messages):
        system = messages[0]['content'] if messages else ''
        return plan if 'json array' in system.lower() or 'planning' in system.lower() else task_result
    return _fn

def _mock_executor(task):
    return 'Result: ' + task.title

# 1. Task + safe_parse_list + parse_plan
t = Task(id='x', title='T', description='d')
assert t.status == 'pending' and t.result == '' and t.depends_on == []
assert safe_parse_list('[1,2,3]') == [1, 2, 3]
assert safe_parse_list('{}') is None and safe_parse_list('no json') is None
tasks = parse_plan(_PLAN_JSON)
assert len(tasks) == 3 and tasks[1].depends_on == ['t1']
assert parse_plan('garbage') == []
print("✅ Task / safe_parse_list / parse_plan")

# 2. topo_sort
ordered = topo_sort(tasks)
ids = [t.id for t in ordered]
assert ids.index('t1') < ids.index('t2') < ids.index('t3')
diamond = [
    Task('base','B','b'), Task('d1','D1','d',depends_on=['base']),
    Task('d2','D2','d',depends_on=['base']),
    Task('final','F','f',depends_on=['d1','d2']),
]
di = topo_sort(diamond); di_ids = [t.id for t in di]
assert di_ids.index('base') < di_ids.index('final')
cycle = [Task('x','X','x',depends_on=['y']), Task('y','Y','y',depends_on=['x'])]
assert len(topo_sort(cycle)) == 2          # cycle guard: both returned
print("✅ topo_sort (chain, diamond, cycle guard)")

# 3. build_execution_context + execute_task
t2 = Task('t2','Step2','do',depends_on=['t1'])
ctx = build_execution_context(t2, {'t1': 'step1 done'})
assert 'step1 done' in ctx and 'Step2' in ctx
assert execute_task(Task('x','X','x'), {}, executor_fn=_mock_executor) == 'Result: X'
assert execute_task(Task('x','X','x'), {}, llm_fn=_mock_planner(task_result='llm reply')) == 'llm reply'
def _bad(t): raise RuntimeError('boom')
assert execute_task(Task('x','X','x'), {}, executor_fn=_bad).startswith('Error:')
print("✅ build_execution_context / execute_task (inject, executor, LLM, exception guard)")

# 4. run_plan
r = run_plan('test', executor_fn=_mock_executor, llm_fn=_mock_planner())
assert len(r['tasks']) == 3 and r['tasks'][0].status == 'done'
assert r['results']['t2'] == 'Result: Draft outline'
assert r['answer'] == 'Result: Write summary'
assert run_plan('x', executor_fn=_mock_executor,
                llm_fn=_mock_planner(plan_json='no json'))['tasks'] == []
print("✅ run_plan (end-to-end, results, answer, empty plan)")

# 5. PlannerAgent
agent = PlannerAgent(executor_fn=_mock_executor, llm_fn=_mock_planner())
plan_tasks = agent.plan('test')
assert len(plan_tasks) == 3 and all(t.status == 'pending' for t in plan_tasks)
results = agent.execute(plan_tasks)
assert plan_tasks[0].status == 'done' and results['t1'] == 'Result: Gather facts'
record = agent.run('another goal')
assert 'goal' in record and record['answer'] == 'Result: Write summary'
assert len(agent.history()) == 1
agent.history().clear(); assert len(agent.history()) == 1  # copy
agent.clear_history(); assert len(agent.history()) == 0
print("✅ PlannerAgent (plan/execute/run/history/clear_history)")

print("\nPlanning agent complete!")
